In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd

ROOT              = Path().resolve().parents[1]
AIRLINES_PATH     = ROOT / "data" / "interim" / "airlines.parquet"
AIRPORTS_PATH     = ROOT / "data" / "interim" / "airports.parquet"
REPORT_PATH       = ROOT / "reports" / "validation_airlines.txt"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

df       = pd.read_parquet(AIRLINES_PATH)
airports = pd.read_parquet(AIRPORTS_PATH)

report_sections = []

def record(section: list, line: str = "") -> None:
    print(line)
    section.append(line)

print("Setup complete.")
print(f"Airlines path: {AIRLINES_PATH}")
print(f"Airports path: {AIRPORTS_PATH}")
print(f"Airlines shape: {df.shape}")

Setup complete.
Airlines path: C:\Users\hp\Desktop\Term 7\Data Science\Project\flight-delay-predictor\data\interim\airlines.parquet
Airports path: C:\Users\hp\Desktop\Term 7\Data Science\Project\flight-delay-predictor\data\interim\airports.parquet
Airlines shape: (5, 4)


In [2]:
section = []
record(section, "=" * 55)
record(section, "CHECK 1 — Shape")
record(section, "=" * 55)
record(section, f"  Rows:    {len(df)}")
record(section, f"  Columns: {df.shape[1]}")

if len(df) == 0:
    record(section, "  FAIL — table is empty")
elif len(df) != 5:
    record(section, f"  WARN — expected 5 rows, got {len(df)}")
else:
    record(section, "  PASS — exactly 5 rows as expected")

report_sections.append(section)

CHECK 1 — Shape
  Rows:    5
  Columns: 4
  PASS — exactly 5 rows as expected


In [3]:
EXPECTED_COLUMNS = [
    "carrier_code",
    "carrier_name",
    "carrier_type",
    "hub_airports",
]

section = []
record(section, "=" * 55)
record(section, "CHECK 2 — Schema")
record(section, "=" * 55)

missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols   = [c for c in df.columns if c not in EXPECTED_COLUMNS]

if missing_cols:
    record(section, f"  FAIL — missing columns: {missing_cols}")
else:
    record(section, f"  PASS — all {len(EXPECTED_COLUMNS)} expected columns present")

if extra_cols:
    record(section, f"  INFO — extra columns found: {extra_cols}")

record(section, "")
record(section, "  Column dtypes:")
for col in df.columns:
    record(section, f"    {col:<22} {str(df[col].dtype)}")

report_sections.append(section)

CHECK 2 — Schema
  PASS — all 4 expected columns present

  Column dtypes:
    carrier_code           str
    carrier_name           str
    carrier_type           str
    hub_airports           str


In [4]:
section = []
record(section, "=" * 55)
record(section, "CHECK 3 — Missing values")
record(section, "=" * 55)

total_missing = df.isnull().sum().sum()

if total_missing == 0:
    record(section, "  PASS — no missing values in any column")
else:
    record(section, f"  FAIL — {total_missing} missing values found:")
    for col in df.columns:
        n = df[col].isnull().sum()
        if n > 0:
            record(section, f"    {col:<22} {n} missing")

report_sections.append(section)

CHECK 3 — Missing values
  PASS — no missing values in any column


In [5]:
section = []
record(section, "=" * 55)
record(section, "CHECK 4 — Duplicates")
record(section, "=" * 55)

exact_dups = df.duplicated().sum()
code_dups  = df.duplicated(subset=["carrier_code"]).sum()

if exact_dups == 0:
    record(section, "  PASS — no exact duplicate rows")
else:
    record(section, f"  FAIL — {exact_dups} exact duplicate rows")

if code_dups == 0:
    record(section, "  PASS — carrier_code is unique (primary key valid)")
else:
    record(section, f"  FAIL — {code_dups} duplicate carrier codes:")
    dupes = df[df.duplicated(subset=["carrier_code"], keep=False)]
    for _, row in dupes.iterrows():
        record(section, f"    {row['carrier_code']}  {row['carrier_name']}")

report_sections.append(section)

CHECK 4 — Duplicates
  PASS — no exact duplicate rows
  PASS — carrier_code is unique (primary key valid)


In [6]:
OUR_CARRIERS = {"AA", "AS", "B6", "DL", "UA"}

section = []
record(section, "=" * 55)
record(section, "CHECK 5 — Carrier code validity")
record(section, "=" * 55)

# Format check — must be exactly 2 uppercase alphanumeric characters
invalid_format = df[
    ~df["carrier_code"].astype(str).str.fullmatch(r"[A-Z0-9]{2}")
]
if len(invalid_format) == 0:
    record(section, "  PASS — all codes are valid 2-character uppercase format")
else:
    record(section, f"  FAIL — {len(invalid_format)} invalid carrier codes:")
    for code in invalid_format["carrier_code"].tolist():
        record(section, f"    - {code}")

# Coverage check — all 5 expected carriers must be present
present = set(df["carrier_code"])
missing = OUR_CARRIERS - present
extra   = present - OUR_CARRIERS

if not missing:
    record(section, "  PASS — all 5 expected carriers present")
else:
    record(section, f"  FAIL — {len(missing)} expected carriers missing:")
    for code in sorted(missing):
        record(section, f"    - {code}")

if extra:
    record(section, f"  WARN — unexpected carriers in table: {sorted(extra)}")

report_sections.append(section)

CHECK 5 — Carrier code validity
  PASS — all codes are valid 2-character uppercase format
  PASS — all 5 expected carriers present


In [7]:
VALID_CARRIER_TYPES = {"Legacy", "Hybrid", "LCC"}

section = []
record(section, "=" * 55)
record(section, "CHECK 6 — Carrier type validity")
record(section, "=" * 55)

invalid_types = df[~df["carrier_type"].isin(VALID_CARRIER_TYPES)]

if len(invalid_types) == 0:
    record(section, "  PASS — all carrier_type values are valid")
else:
    record(section, f"  FAIL — {len(invalid_types)} invalid carrier_type values:")
    for _, row in invalid_types.iterrows():
        record(section, f"    {row['carrier_code']}  '{row['carrier_type']}'")

record(section, "")
record(section, "  Carrier type distribution:")
for ctype, count in df["carrier_type"].value_counts().items():
    record(section, f"    {ctype:<10} {count}")

report_sections.append(section)

CHECK 6 — Carrier type validity
  PASS — all carrier_type values are valid

  Carrier type distribution:
    Legacy     4
    Hybrid     1


In [8]:
section = []
record(section, "=" * 55)
record(section, "CHECK 7 — Hub airports cross-table consistency")
record(section, "=" * 55)
record(section, "  Verifying every hub airport exists in airports.parquet ...")
record(section, "")

airport_codes_in_metadata = set(airports["airport_code"])
all_passed = True

for _, row in df.iterrows():
    carrier  = row["carrier_code"]
    hubs     = [h.strip() for h in row["hub_airports"].split(",")]
    missing  = [h for h in hubs if h not in airport_codes_in_metadata]

    if missing:
        record(section, f"  WARN — {carrier} has hubs not in airports table: {missing}")
        record(section, f"         These will produce NULLs in hub feature engineering")
        all_passed = False
    else:
        record(section, f"  PASS — {carrier}: all {len(hubs)} hubs found in airports table")

record(section, "")
if all_passed:
    record(section, "  PASS — all hub airports covered by airports metadata table")
else:
    record(section, "  WARN — some hub airports missing from airports metadata table")
    record(section, "         Consider adding them to OUR_AIRPORTS in acquire_airports.py")

report_sections.append(section)

CHECK 7 — Hub airports cross-table consistency
  Verifying every hub airport exists in airports.parquet ...

  PASS — AA: all 8 hubs found in airports table
  PASS — AS: all 5 hubs found in airports table
  PASS — B6: all 5 hubs found in airports table
  PASS — DL: all 8 hubs found in airports table
  PASS — UA: all 6 hubs found in airports table

  PASS — all hub airports covered by airports metadata table


In [9]:
section = []
record(section, "=" * 55)
record(section, "CHECK 8 — Full table display")
record(section, "=" * 55)
record(section, "")

for _, row in df.iterrows():
    record(section, f"  carrier_code : {row['carrier_code']}")
    record(section, f"  carrier_name : {row['carrier_name']}")
    record(section, f"  carrier_type : {row['carrier_type']}")
    record(section, f"  hub_airports : {row['hub_airports']}")
    record(section, "")

report_sections.append(section)

CHECK 8 — Full table display

  carrier_code : AA
  carrier_name : American Airlines Inc.
  carrier_type : Legacy
  hub_airports : DFW,CLT,MIA,ORD,PHX,JFK,LAX,PHL

  carrier_code : AS
  carrier_name : Alaska Airlines Inc.
  carrier_type : Legacy
  hub_airports : SEA,PDX,SFO,LAX,ANC

  carrier_code : B6
  carrier_name : JetBlue Airways
  carrier_type : Hybrid
  hub_airports : BOS,JFK,FLL,LAX,MCO

  carrier_code : DL
  carrier_name : Delta Air Lines Inc.
  carrier_type : Legacy
  hub_airports : ATL,DTW,MSP,SLC,SEA,BOS,JFK,LAX

  carrier_code : UA
  carrier_name : United Air Lines Inc.
  carrier_type : Legacy
  hub_airports : ORD,EWR,IAH,DEN,SFO,LAX



In [10]:
header = [
    "=" * 55,
    "AIRLINES METADATA VALIDATION REPORT",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"File:      {AIRLINES_PATH}",
    f"Rows:      {len(df)}",
    f"Columns:   {df.shape[1]}",
    "=" * 55,
    "",
]

all_lines = header.copy()
for section in report_sections:
    all_lines.extend(section)
    all_lines.append("")

REPORT_PATH.write_text("\n".join(all_lines), encoding="utf-8")

print("=" * 55)
print("Report written successfully.")
print(f"Location: {REPORT_PATH}")
print(f"Total checks: {len(report_sections)}")
print("=" * 55)

Report written successfully.
Location: C:\Users\hp\Desktop\Term 7\Data Science\Project\flight-delay-predictor\reports\validation_airlines.txt
Total checks: 8
